## find the projected velocity of viable shredded galaxies
the point is to reduce number of viable galaxies before VI

In [1]:
from astropy.table import Table
from astropy.constants import c
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.visualization.wcsaxes import SphericalCircle

from get_cutouts import get_cutout

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from tqdm import tqdm

from galaxy_selection import select_galaxy, velocity_selection

from velocity_map_fxns import sin_i, criteria, criteria_sym

In [2]:
loa_table = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_v2.fits')
# loa_table = Table.read('/pscratch/sd/d/dbustos/rot_curves/loa_targs_v2.fits')

loa_table[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64
6521555517440,1157225,204.22749,32.09493,0.010016011261740405,0.010016011261740405,0,71.96394567680545,0.002217102021260331,0.36627155666222244,65.12960815429688,114.35180431415512,49.22219615985824,1
46529817608200,1026585,262.7290855,56.8275909,0.030065414133491536,0.030065414133491536,0,9134.873342573643,0.000960130367504636,0.20759597541001423,50.010528564453125,232.32086291334522,182.3103343488921,1
50491463565312,832982,186.082682159506,31.514254503332,0.0039163048042792185,0.0039163048042792185,0,53.936871471232735,--,--,--,--,--,1
50532307697666,356915,186.497709668916,33.6029271154256,0.001097729790163198,0.001097729790163198,0,4260.606228693388,--,--,--,--,--,1
67792481026048,992860,129.121654946794,18.365939088565412,1.0791507885943028,1.0791507885943028,4,3.1017794758081436,0.006014833578063982,0.8697308071747932,33.59540939331055,32.12214705731604,1.4732623359945052,1


In [3]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')
SGA[:5]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
2,SGA-2020 2,PGC1283207,1283207,228.3770865,5.4232017,S?,152.2,0.36307806,0.724436,0.03463229,23.40448,16.976,False,LEDA-20181114,0,PGC1283207,1,True,228.3770865,5.4232017,0.36307806,2283p055,228.3770803831908,5.423191398593787,0.49470574,SB26,158.20142,0.545691,228.37700918822188,5.4232652570544015,10.897086,3.3509698,3.1147978,3.240862,5.902337,6.9126143,7.941369,8.997992,10.073601,11.199986,12.391357,13.561038,14.841172,16.966799,16.108246,15.486356,16.879545,16.024958,15.400715,16.818878,15.967034,15.341793,16.776297,15.925804,15.300776,16.746685,15.897334,15.272053,16.725166,15.876816,15.2521105,16.708357,15.862035,15.237181,16.696539,15.851936,15.226998,16.689613,15.844313,15.21976,0.013392451,0.02354,0.021872982,0.01736985,0.024445537,0.039866067,0.05026544,0.08455789,0.122911856,0.005682776,0.0054258136,0.0049038026,0.005588406,0.005323561,0.0047632363,0.00543534,0.005177031,0.0046343105,0.0053025587,0.005040888,0.0045181247,0.005206092,0.0049438984,0.0044374703,0.0051483097,0.0048758644,0.0043834248,0.0051032505,0.0048264163,0.004344248,0.0050705094,0.004792021,0.004319857,0.005054293,0.004765629,0.0043044444,16.65942,0.34037337,0.2978292,3.0239506,0.07928849,15.820566,0.2640441,0.34559453,3.3033552,0.003811298,15.195567,0.29826432,0.3001073,3.2333765,0.011723555,0
3,SGA-2020 3,PGC1310416,1310416,202.54443750000002,6.9345944,Sc,159.26,0.4017908,0.7816278,0.073888786,23.498482,16.85,False,LEDA-20181114,1,PGC1310416,1,True,202.54443750000002,6.9345944,0.4017908,

In [4]:
SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

In [5]:
len(loa_table)

115256

## calculate rotational velocity (without deprojecting)
if a fiber is >1000 km/s before deprojection, it will probably be >1000 km/s after

In [6]:
# new error correcting for redrock 7km/s systematic uncertainty
dv_sys = 7 #km/s
dz_sys = dv_sys/c.to('km/s').value
loa_table['ZERR_MOD'] = np.sqrt(loa_table['Z_ERR']**2 + (dz_sys*(1 + loa_table['Z']))**2)

In [7]:
zwarn = 0
deltachi2 = 25

center_dist_lim = 0.001
unique_dist_lim = 0.01

q0 = 0.2

In [8]:
loa = loa_table

In [9]:
loa['unique_obs'] = np.nan

for i in tqdm(np.unique(loa['SGA_ID'])):
    
    fiber = loa['SGA_ID'] == i 

    obs = loa[fiber]

    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]
    if len(on_major) == 0:
        unique_on = 0

    else:
        on_major['DIST_R26'] = np.round(on_major['DIST_R26'], 6)
        on_major = on_major.group_by('DIST_R26')
        unique_on = len(on_major.groups)

    # identify all points off major axis
    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]
    
    # find number of unique points off axis
    num_off_major = len(off_major)

    if num_off_major == 0:
        unique_off = 0
    else:
        off_major = off_major.group_by(['DIST_R26','ANGLE_OFF_AXIS'])
        unique_off = len(off_major.groups)

    total_obs = unique_on + unique_off

    loa['unique_obs'][fiber] = total_obs

100%|██████████| 21443/21443 [01:39<00:00, 215.22it/s]


In [10]:
loa['Velocity'] = np.nan
loa['V_err'] = np.nan
loa['Z_center'] = np.nan
#loa['c_or_s'] = 0

c = c.to('km/s')

for i in tqdm(np.unique(loa['SGA_ID'])):
    has_valid_center = False
    
    fiber = loa['SGA_ID'] == i 

    obs = loa[fiber]

    #find the index for this target in SGA
    #sga_idx = SGA_dict[i]
    
    # find z and its error for all targets
    z_targ = loa['Z'][fiber]
    z_e = loa['ZERR_MOD'][fiber]

    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]

    total_obs = obs['unique_obs'][0]

#-----------------------------------------------------------------------
# find center redshift if there is a center target that meets criteria
#-----------------------------------------------------------------------
    
    # check for center fiber
    if np.any(on_major['DIST_R26'] < center_dist_lim):
        
        # find all center fibers that fit quality check
        criteria_c = (on_major['DIST_R26'] < center_dist_lim)
        center = on_major[criteria_c & criteria_sym(on_major, zwarn, deltachi2)]

        # find z for center targets
        z_c = center['Z']

        # calculate weight of center targets
        zc_err = center['ZERR_MOD']
        weight = 1/(zc_err ** 2)
        #print(weight)
                
        # get weighted average
        # if weight = 0, then it did not meet criteria. 
        if np.any(weight > 0):
            has_valid_center = True
            z_cen = np.average(z_c, weights = weight)
            z_cen_err = np.sqrt(np.sum(zc_err ** 2))
    
#--------------------------------------------------------------
# find the center redshift given two symmetric points
#--------------------------------------------------------------
    if not has_valid_center:

        sga_id = SGA_dict[i]

        #split targets on semimajor axis onto either side of center galaxy
        if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

            left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

        else:
            left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
            
        left = on_major[left_index]
        right = on_major[~left_index]

        #----------------------------------------------------------
        # identify where the symmetric points are and group them ------------
        #----------------------------------------------------------
        #find R26 for each fiber
        right_dist, left_dist = np.array(right['DIST_R26']), np.array(left['DIST_R26'])

        #create a matrix subtracting each right r26 element from each left r26 element
        diff_matrix = np.abs(right_dist[:,np.newaxis]-left_dist)

        #identify all points in matrix where difference is within unique dist (fibers are symmetric)
        right_idx, left_idx = np.where(diff_matrix < unique_dist_lim)

        #make sure there are symmetric points
        if (len(right_idx) == 0) or (len(left_idx) == 0):  
            
            # if not, check if there are 10 unique points
            if total_obs >= 10:
                # get the average redshift
                weight = 1/(z_e **2)
                z_cen = np.average(z_targ, weights = weight)
                z_cen_err = np.sqrt(np.sum(z_e**2))
                
            else:
                print(sga_id, 'no center, symmetric point, or random pts')
                continue  
        
        elif (len(right_idx) != 0) and (len(left_idx) != 0):
    
            #put those fibers into appropriate tables
            right['DIST_R26'] = np.round(right['DIST_R26'],6)
            left['DIST_R26'] = np.round(left['DIST_R26'],6)
            
            symmetric_right = right[np.unique(right_idx)].group_by('DIST_R26')
            symmetric_left = left[np.unique(left_idx)].group_by('DIST_R26')
    
            # print('left: ', symmetric_left['DIST_R26'], len(symmetric_left.groups))
            # print('right:', symmetric_right['DIST_R26'], len(symmetric_right.groups))
            
    #--------------------------------------
    # get pseudo-center for each grouping
    #--------------------------------------
            # number of fiber groups
            len_sym = len(symmetric_right.groups)
      
            #empty array for z_c and weight for each fiber group
            z_c = np.empty(len_sym)
            weight = np.empty(len_sym)
            
        #-----------------------   
        # for each fiber group
        #-----------------------
            for z in range(len_sym):
                
                #-----------------------------------------
                # get right points
                #-----------------------------------------
                right_group = symmetric_right.groups[z]
                
                # check if there is more than one observation
                if len(right_group) > 1:
                    # if there is, only take the good observations
                    if np.any(right_group['ZWARN'] == 0) and np.any(right_group['DELTACHI2'] > 25):
                        right_group = right_group[criteria_sym(right_group, zwarn, deltachi2)]
                
                #-----------------------------------------
                # get left points
                #-----------------------------------------
                left_group = symmetric_left.groups[z]
                
                # check if there is more than one observation
                if len(left_group) > 1:
                    # if there is, only take the good observations
                    if np.any(left_group['ZWARN'] == 0) and np.any(left_group['DELTACHI2'] > 25):
                        left_group = left_group[criteria_sym(left_group, zwarn, deltachi2)]
                        
                #--------------------------------
                # get redshift and pseudo center
                #--------------------------------
                # get average z for right and left
                z_right = right_group['Z'].mean()
                z_left = left_group['Z'].mean()
    
                # pseudo z_center for fiber group
                z_c[z] = (z_right+z_left)/2
    
                # propagate uncertainty
                zr_err = np.sum(right_group['ZERR_MOD'] ** 2)
                zl_err = np.sum(left_group['ZERR_MOD'] ** 2)
                
                z_err = np.sqrt(zr_err + zl_err)/2
              
                # weight for each fiber group
                weight[z] = 1/(z_err ** 2)
    
        #----------------------------------------------------
        # if multiple symmetric groups, remove the bad pairs
        #----------------------------------------------------
            if len_sym > 1:
                
                # copy array of z_c and weight
                zc_pairs = z_c.copy()
                weight_pairs = weight.copy()
                
                for pair in range(len_sym):
                    right_group = symmetric_right.groups[pair]
                    right_group = right_group[criteria_sym(right_group, zwarn, deltachi2)]
                
                    left_group = symmetric_left.groups[pair]
                    left_group = left_group[criteria_sym(left_group, zwarn, deltachi2)]
    
                    # if all fibers in the group are bad, then remove it from redshift and weight calculation
                    
                    if (len(right_group) == 0) or (len(left_group) == 0):
                        zc_pairs[pair] = 0
                        weight_pairs[pair] = 0
                        
                # if every pair is bad, then keep all the pairs
                if np.sum(zc_pairs) == 0:
                    print(i,'all bad pairs zc')
                    z_c = z_c
                else:
                    z_c = zc_pairs
    
                if np.sum(weight_pairs) == 0:
                    print(i,'all bad pair weight')
                    weight = weight
                else:
                    weight = weight_pairs 
    
        #-----------------------------------------------------
        # calculate weighted pseudo z_center and error
        #-----------------------------------------------------
            z_cen = np.average(z_c, weights = weight)
            
            z_cen_err = np.sqrt(np.sum(z_err**2))
    
            # print('pseudo:',z_cen)
    
#---------------------------------------------------------
#find the redshift of each fiber relative to the center
#---------------------------------------------------------

    # relative redshift
    z_rel = (1 + z_targ)/(1 + z_cen) - 1

    #inclination angle
    # axis_ratio = SGA['BA'][sga_idx]
    # inc = sin_i(axis_ratio, q0)
    
    # find the rotational velocity
    velocity = z_rel*c

    # rotational velocity error
    v_error = c*np.sqrt((z_cen_err**2)+(z_e**2))

    counter = np.where(abs(velocity.value) < 1000, 1, 0)
   
    if np.sum(counter) < 3:
        #print (sga_idx, ':not enough points')
        loa['Velocity'][fiber] = np.nan
        loa['V_err'][fiber] = np.nan

    elif np.sum(counter) >= 3:
        loa['Velocity'][fiber] = velocity  

        loa['V_err'][fiber] = v_error
    
        loa['Z_center'][fiber] = z_cen_err
    #print(velocity)

  1%|          | 203/21443 [00:00<01:30, 233.84it/s]

2402 no center, symmetric point, or random pts


  1%|▏         | 284/21443 [00:01<01:22, 255.02it/s]

3335 no center, symmetric point, or random pts


  4%|▍         | 824/21443 [00:03<01:15, 273.10it/s]

...

377931 no center, symmetric point, or random pts
378120 no center, symmetric point, or random pts
378178 no center, symmetric point, or random pts


100%|█████████▉| 21388/21443 [01:33<00:00, 180.61it/s]

2000055 all bad pairs zc
2000055 all bad pair weight
380558 no center, symmetric point, or random pts


100%|██████████| 21443/21443 [01:34<00:00, 227.84it/s]

382436 no center, symmetric point, or random pts


In [11]:
# create a new table that removes all NAN
tf_loa = loa[~np.isnan(loa['Velocity'])]
print(len(np.unique(tf_loa['SGA_ID'])))

10764


In [12]:
tf_loa.write('/pscratch/sd/d/dbustos/rot_curves/loa_targs.fits',format='fits',overwrite=True)